# TMJDLC — Javtiful Download Test

This notebook tests the Javtiful provider using the shared `scripts/detect_stream.py` browser detector. It activates the player using the verified play-gate sequence, prefers the first-party Javtiful `/playback` API source over unrelated ad/player media requests, and redacts signed video source URLs from reports. GitHub Actions and Colab use the same detector. The capture window defaults to 15 seconds, matching the working reference notebook.


In [ ]:
# Install/update TMJDLC and Chromium
%cd /content
!if [ -d /content/TMJDLC/.git ]; then git -C /content/TMJDLC pull --ff-only; else git clone --depth 1 https://github.com/Telebotfaroff/TMJDLC.git /content/TMJDLC; fi
%cd /content/TMJDLC
!python -m pip install --upgrade pip
!python -m pip install -e .
!python -m playwright install --with-deps chromium
!sudo apt-get update -qq && sudo apt-get install -y ffmpeg

In [ ]:
#@title TMJDLC build verification — check this before testing
TMJDLC_COLAB_BUILD = "TMJDLC-COLAB-DEV-2026-10-31-PLAYBACK-SOURCE-FIX"
import subprocess
from pathlib import Path

repo = Path('/content/TMJDLC')
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
detector = repo / 'scripts' / 'detect_stream.py'
provider_detector = repo / 'tmjdlc' / 'providers' / 'javtiful' / 'stream_detector.py'
assert detector.is_file(), 'Shared Actions detector script is missing.'
assert provider_detector.is_file(), 'Javtiful provider detector is missing.'
detector_source = detector.read_text(encoding='utf-8')
provider_source = provider_detector.read_text(encoding='utf-8')
assert '--resolved-output' in detector_source, 'Old detector version: resolved-output support is missing.'
assert 'scripts/detect_stream.py' in provider_source, 'Provider is not using the shared detector.'
assert 'playback_sources' in detector_source and 'selected_playback_source' in detector_source, 'First-party playback API source selection is missing.'
assert 'pkey' in detector_source and 'current_src' in detector_source, 'Signed URL redaction safeguards are missing.'
assert 'block_external_top_navigation' in detector_source, 'Pop-under navigation guard is missing.'
assert 'External media requests were discarded.' in detector_source, 'External-page media rejection is missing.'
print('BUILD MARKER:', TMJDLC_COLAB_BUILD)
print('CHECKED-OUT COMMIT:', commit)
print('STATUS: playback API selection and signed-source redaction checks passed')


## 1. Configure the download

In [ ]:
#@title Javtiful download settings
VIDEO_URL = "https://javtiful.com/video/115269/miab-682-reducing-mosaic" #@param {type:"string"}
QUALITY = "best" #@param ["best", "1080", "720", "480", "360"]
OUTPUT_DIR = "/content/tmjdlc-downloads" #@param {type:"string"}
MAX_ATTEMPTS = 2 #@param {type:"integer"}

from pathlib import Path
from urllib.parse import urlsplit
if urlsplit(VIDEO_URL).hostname not in {"javtiful.com", "www.javtiful.com"}:
    raise ValueError("VIDEO_URL must be an original javtiful.com video page URL")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
quality_value = None if QUALITY == "best" else int(QUALITY)
print(f"URL: {VIDEO_URL}")
print(f"Quality: {QUALITY}")
print(f"Maximum fresh-resolution attempts: {MAX_ATTEMPTS}")

## 2. Resolve and download
The provider performs playback resolution immediately before each download attempt. Signed media URLs remain in memory only; the displayed stream report redacts sensitive query parameters.

In [ ]:
import asyncio
import json
import re
import time
from tmjdlc.providers.javtiful import JavtifulProvider

video_match = re.search(r"/video/(\d+)", VIDEO_URL)
video_id = video_match.group(1) if video_match else "video"
filename = f"javtiful-{video_id}"

def report_progress(message):
    print(f"[TMJDLC] {message}")

provider = JavtifulProvider(
    max_attempts=MAX_ATTEMPTS,
    progress_callback=report_progress,
)
started = time.monotonic()
# Playwright's synchronous API must run outside Colab's active asyncio loop.
downloaded_path, stream = await asyncio.to_thread(
    provider.download,
    VIDEO_URL.strip(),
    OUTPUT_DIR,
    filename,
    quality=quality_value,
)
elapsed = time.monotonic() - started
size = downloaded_path.stat().st_size
if size <= 0:
    raise RuntimeError("Downloaded file is empty")

download_result = {
    "status": "downloaded",
    "filename": downloaded_path.name,
    "size_bytes": size,
    "size_mib": round(size / (1024 ** 2), 2),
    "elapsed_seconds": round(elapsed, 2),
    "average_mib_per_second": round(size / (1024 ** 2) / elapsed, 3) if elapsed else None,
    "stream": stream.to_public_dict(),
}
print("\nDOWNLOAD RESULT")
print(json.dumps(download_result, ensure_ascii=False, indent=2))

## 3. Validate with FFprobe

In [ ]:
import subprocess
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=format_name,duration,size:stream=codec_type,codec_name,width,height",
     "-of", "json", str(downloaded_path)],
    capture_output=True, text=True,
)
if probe.returncode != 0:
    print(probe.stderr[-4000:])
    raise RuntimeError("FFprobe could not validate the downloaded file")
probe_data = json.loads(probe.stdout)
print(json.dumps(probe_data, indent=2))

## 4. Download the completed file to your device

In [ ]:
from google.colab import files
files.download(str(downloaded_path))

## 5. Send the test results
If the run fails, share the progress output and traceback. The safe stream report masks signed query values. Never share cookies, authorization headers, or a raw signed media URL.